# Interpolation of Functions

_____________________________________________________________________

### The Interpolation Problem
Given $n+1$ points $$(x_0,y_0),...,(x_n,y_n)\in \mathbb{R}^2$$ with pairwise different $x_0,..,x_n$, find a a Polynom of degree $n$, that is from the set $$\Pi_n:=\{p(x)=\sum_{i=0}^na_ix^i: a_i \in \mathbb{R}, i=0,...,n\}$$ such that $$y_i=p(x_i) \ \ \ \forall i\in\{0,...,n\}, p\in \Pi_n.$$

________________________________________________________________

### Lagrange's Interpolation
##### Lagrange Basis Polynomials
For $n+1$ paarwise different $x_0,...,x_n\in\mathbb{R}$ **define** Polynomials $L_0,...,L_k \in \Pi_n$ with $$L_k(x)=\prod_{i=0, i\neq k}^n \frac{x-x_i}{x_k-x_i},  k=0,...,n$$

These Polynomials have the following Properties:
- $L_k(x_j)=\delta_{kj}$
- form a basis for the vector space $\Pi_n$

##### Lagrange's Interpolation formula
Let everything be given as in the interpolation problem. Then **there exists exactly one** interpolating Polynom $\pi \in \Pi_n$ like in the interpolation problem and it has the form $$p(x)=\sum_{k=0}^n y_k L_k(x)$$

Proof Idea:
- Existence: calculate the polynom
- Uniqueness: Assume there is a similar $q$, show that $p-q$ has a degree $\leq n $ but $n+1$ zeroes.

##### Cons of Lagrange's Interpolation
- Instabilities when calculating fractions, specially when calculating values of $p(x)$ with $x$ being close to $x_j$
- Slow ( $O(2n^2)$)
- Inflexible. Is a new ppoint added, so the polynomial has to be calculated again.

_____________________________________________________________________
### The Neville-Schema

##### Notation
Let $k,m\in\mathbb{N} \cup \{0\}, (x_{k},y_{k}),...,(x_{k+m},y_{k+m})\in \mathbb{R}^2$, then let $p_{k,k+1,...,k+m}$ be the unique polynomial of degree $\leq m$, such that $$p_{k,k+1,...,k+m}(x_j)=y_j $$ for all $ \ j=k,k+1,...,k+m$. That is, $p_{k,k+1,...,k+m}$ is the interpolating polynomial of points $(x_{k},y_{k}),...,(x_{k+m},y_{k+m})$


##### Recursion
Let everything be given as in *the inteprolation problem* and $k,m \geq 0$ $ k+m \leq n $. Then
- $$p_k(x)=y_k$$
- $$p_{k,k+1,...,k+m}(x)= \frac{ (x-x_k)p_{k+1,...,k+m}(x)-(x-x_{k+m})p_{k,...,k+m-1}(x) }{x_{k+m}-x_k}$$

##### Python Implementation 1

In [39]:
import sympy as sp

class Point:
    def __init__(self, x:float, y:float):
        self.x = x
        self.y= y

def read_points(n:int) -> list:
    points = []
    for i in range(n):
        points.append( Point(float(input(f"\nx_{i}=")), float(input(f"y_{i}="))) )
    return points


def p(k:int, m:int, points:list[Point], x:sp.symbols):
    """
        Calculates interpolating polynomial for {points} indexed from k up to k+m recursively.
    """
    if m==0:
        return sp.Rational(points[k].y)
    return ( (x-points[k].x)*p(k+1,m-1, points, x) - (x - points[k+m].x)*p(k, m-1, points, x) )/(points[k+m].x-points[k].x)


def interpolate(n:int):
    k = 0
    m = n - 1
    points = read_points(n)
    x = sp.symbols("x")
    
    pol = sp.Poly(p(k,m,points, x).expand())
    sp.pretty(pol)
    return pol.all_coeffs()
    
interpolate(3)


x_0= 0
y_0= 1

x_1= 1
y_1= 3

x_2= 3
y_2= 2


[-0.833333333333333, 2.83333333333333, 1.00000000000000]

The problem with this implementation is that it uses $2^k$ steps. Each recursion calculates values that have been calculated before again. The solution is to store them in a matrix. Our matrix $A$ will have a shape $n\times m$ where an entry $A_{km}$ will refer to $p_{k,...,k+m}$, 

    

##### Python implementation 2: Dynamic programming

$O(n^2)$

In [ ]:
import sympy as sp
import numpy as np

class Point:
    def __init__(self, x:float, y:float):
        self.x = x
        self.y= y

def read_points(n:int) -> list:
    points = []
    for i in range(n):
        points.append( Point(float(input(f"\nx_{i}=")), float(input(f"y_{i}="))) )
    return points


def p(k:int, m:int, points:list[Point], x:sp.symbols, values:np.ndarray):
    """
        Calculates interpolating polynomial for {points} indexed from {k} up to {k+m} recursively.
        For the values that have been calculated before gets then from the {values} Matrix.
    """
    if values[k][m]!=0:
        return sp.Rational(values[k][m])
    if m==0:
        return sp.Rational(points[k].y)
    pp = ( (x-points[k].x)*p(k+1,m-1, points, x, values) - (x - points[k+m].x)*p(k, m-1, points, x, values) )/(points[k+m].x-points[k].x)
    values[k][m] = pp
    return pp


def interpolate(n:int):
    k = 0
    m = n - 1
    points = read_points(n)
    x = sp.symbols("x")
    values = np.zeros(shape=(n,n), dtype=sp.core.add.Add)
    
    pol = sp.Poly(p(k,m,points, x, values).expand())
    sp.pretty(pol)
    return pol.all_coeffs()
    
interpolate(3)


x_0= 0
y_0= 1

x_1= 1
y_1= 3

x_2= 3
y_2= 2


[-0.833333333333333, 2.83333333333333, 1.00000000000000]